# Advanced Analytics

This notebook covers tail risk, rolling Sharpe, investor cohorts, SIP continuity, risk-appetite recommendations, and equity portfolio concentration for the 40 mutual fund schemes.

In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

TRADING_DAYS = 252

# ----------------------------------------------------
# Find the project root automatically
# ----------------------------------------------------
ROOT = Path.cwd()

while not (ROOT / "data").exists():
    if ROOT.parent == ROOT:
        raise FileNotFoundError(
            "Could not find the 'data' folder. "
            "Please ensure your project structure is correct."
        )
    ROOT = ROOT.parent

DATA_DIR = ROOT / "data" / "processed"

print("Project Root :", ROOT)
print("Data Folder  :", DATA_DIR)

# ----------------------------------------------------
# Verify required files exist
# ----------------------------------------------------
required_files = [
    "cleaned_nav_history.csv",
    "cleaned_fund_master.csv",
    "cleaned_investor_transactions.csv",
    "cleaned_portfolio_holdings.csv",
    "fund_scorecard.csv"
]

print("\nChecking files...")
for file in required_files:
    path = DATA_DIR / file
    if path.exists():
        print(f"✓ {file}")
    else:
        raise FileNotFoundError(f"Missing file: {path}")

# ----------------------------------------------------
# Load datasets
# ----------------------------------------------------
nav = pd.read_csv(
    DATA_DIR / "cleaned_nav_history.csv",
    parse_dates=["date"]
)

funds = pd.read_csv(
    DATA_DIR / "cleaned_fund_master.csv",
    parse_dates=["launch_date"]
)

txns = pd.read_csv(
    DATA_DIR / "cleaned_investor_transactions.csv",
    parse_dates=["transaction_date"]
)

holdings = pd.read_csv(
    DATA_DIR / "cleaned_portfolio_holdings.csv",
    parse_dates=["portfolio_date"]
)

scorecard = pd.read_csv(
    DATA_DIR / "fund_scorecard.csv"
)

# ----------------------------------------------------
# Convert AMFI codes to integers
# ----------------------------------------------------
for df in [nav, funds, txns, holdings, scorecard]:
    if "amfi_code" in df.columns:
        df["amfi_code"] = (
            pd.to_numeric(df["amfi_code"], errors="coerce")
            .dropna()
            .astype(int)
        )

# ----------------------------------------------------
# Create daily returns matrix
# ----------------------------------------------------
returns = (
    nav.sort_values(["date", "amfi_code"])
       .pivot(index="date", columns="amfi_code", values="nav")
       .pct_change()
)

print("\nData Loaded Successfully")
print(f"Number of schemes : {returns.shape[1]}")
print(f"Number of dates   : {returns.shape[0]}")
print(f"Start Date        : {returns.index.min().date()}")
print(f"End Date          : {returns.index.max().date()}")

print("\nNAV Shape:", nav.shape)
print("Funds Shape:", funds.shape)
print("Transactions Shape:", txns.shape)
print("Holdings Shape:", holdings.shape)
print("Scorecard Shape:", scorecard.shape)

Project Root : d:\Intern_BlueStock\MutualFundAnalytics
Data Folder  : d:\Intern_BlueStock\MutualFundAnalytics\data\processed

Checking files...
✓ cleaned_nav_history.csv
✓ cleaned_fund_master.csv
✓ cleaned_investor_transactions.csv
✓ cleaned_portfolio_holdings.csv
✓ fund_scorecard.csv

Data Loaded Successfully
Number of schemes : 40
Number of dates   : 1150
Start Date        : 2022-01-03
End Date          : 2026-05-29

NAV Shape: (46000, 3)
Funds Shape: (40, 15)
Transactions Shape: (32778, 13)
Holdings Shape: (322, 8)
Scorecard Shape: (40, 27)


## Historical VaR and CVaR

In [3]:
var_cvar = []
for amfi_code in returns.columns:
    r = returns[amfi_code].dropna()
    var_95 = r.quantile(0.05)
    cvar_95 = r[r <= var_95].mean()
    var_cvar.append({"amfi_code": int(amfi_code), "observations": len(r), "var_95_daily_return": var_95, "cvar_95_daily_return": cvar_95, "var_95_pct": var_95 * 100, "cvar_95_pct": cvar_95 * 100})

var_cvar_report = pd.DataFrame(var_cvar).merge(
    funds[["amfi_code", "scheme_name", "fund_house", "category", "sub_category", "risk_category"]],
    on="amfi_code",
    how="left",
).sort_values("var_95_daily_return")
var_cvar_report.to_csv(ROOT / "var_cvar_report.csv", index=False)
var_cvar_report.head(10)

,amfi_code,observations,var_95_daily_return,cvar_95_daily_return,var_95_pct,cvar_95_pct,scheme_name,fund_house,category,sub_category,risk_category
22,119599,1149,-0.026859,-0.032384,-2.685944,-3.238412,SBI Small Cap Fund - Direct Plan - Growth,SBI Mutual Fund,Equity,Small Cap,Very High
17,119095,1149,-0.026188,-0.031667,-2.618842,-3.166729,Axis Small Cap Fund - Regular - Growth,Axis Mutual Fund,Equity,Small Cap,Very High
4,101207,1149,-0.026021,-0.032459,-2.602125,-3.245906,ABSL Small Cap Fund - Regular - Growth,Aditya Birla Sun Life MF,Equity,Small Cap,Very High
11,118634,1149,-0.025438,-0.032304,-2.543811,-3.230407,Nippon India Small Cap Fund - Regular - Growth,Nippon India MF,Equity,Small Cap,Very High
21,119598,1149,-0.024507,-0.030595,-2.450705,-3.059526,SBI Small Cap Fund - Regular Plan - Growth,SBI Mutual Fund,Equity,Small Cap,Very High
39,149324,1149,-0.023483,-0.031036,-2.348307,-3.103625,DSP Small Cap Fund - Regular - Growth,DSP Mutual Fund,Equity,Small Cap,Very High
7,102886,1149,-0.019220,-0.023251,-1.922028,-2.325086,UTI Mid Cap Fund - Regular - Growth,UTI Mutual Fund,Equity,Mid Cap,High
2,100033,1149,-0.019034,-0.023456,-1.903354,-2.345576,HDFC Mid-Cap Opportunities Fund - Regular - Gr...,HDFC Mutual Fund,Equity,Mid Cap,High
25,120505,1149,-0.018892,-0.024342,-1.889179,-2.434207,ICICI Pru Midcap Fund - Regular - Growth,ICICI Prudential MF,Equity,Mid Cap,High
16,119094,1149,-0.018480,-0.024260,-1.848028,-2.426006,Axis Midcap Fund - Regular - Growth,Axis Mutual Fund,Equity,Mid Cap,High


## Rolling 90-Day Sharpe

In [4]:
top_codes = scorecard.sort_values("score_rank").head(5)["amfi_code"].astype(int).tolist()
rolling_sharpe = returns[top_codes].rolling(90).mean() / returns[top_codes].rolling(90).std() * np.sqrt(TRADING_DAYS)
names = scorecard.set_index("amfi_code")["scheme_name"].to_dict()

sns.set_theme(style="whitegrid", palette="Set2")
fig, ax = plt.subplots(figsize=(14, 7))
for amfi_code in top_codes:
    ax.plot(rolling_sharpe.index, rolling_sharpe[amfi_code], label=names.get(amfi_code, str(amfi_code)).replace(" - Growth", ""))
ax.axhline(0, color="#555555", linewidth=0.9, alpha=0.7)
ax.set_title("Rolling 90-Day Sharpe Ratio: Top 5 Scorecard Funds")
ax.set_xlabel("Date")
ax.set_ylabel("Annualized Sharpe")
ax.legend(loc="best", fontsize=8)
fig.tight_layout()
fig.savefig(ROOT / "rolling_sharpe_chart.png", dpi=160)
plt.show()

C:\Users\panal\AppData\Local\Temp\ipykernel_20076\1140213887.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Investor Cohort Analysis

In [5]:
tx = txns.copy()
tx["txn_type_norm"] = tx["transaction_type"].str.lower().str.replace(" ", "_")
tx["is_investment"] = tx["txn_type_norm"].isin(["sip", "purchase", "lumpsum", "switch_in"])
tx["is_sip"] = tx["txn_type_norm"].eq("sip")
tx["first_transaction_year"] = tx.groupby("investor_id")["transaction_date"].transform("min").dt.year

cohorts = tx.groupby("first_transaction_year").agg(
    investor_count=("investor_id", "nunique"),
    avg_sip_amount=("amount_inr", lambda s: s[tx.loc[s.index, "is_sip"]].mean()),
    total_invested=("amount_inr", lambda s: s[tx.loc[s.index, "is_investment"]].sum()),
).reset_index()

top_funds = (
    tx[tx["is_investment"]]
    .groupby(["first_transaction_year", "amfi_code"], as_index=False)["amount_inr"]
    .sum()
    .sort_values(["first_transaction_year", "amount_inr"], ascending=[True, False])
    .drop_duplicates("first_transaction_year")
    .merge(funds[["amfi_code", "scheme_name"]], on="amfi_code", how="left")
)
cohorts = cohorts.merge(top_funds[["first_transaction_year", "scheme_name", "amount_inr"]], on="first_transaction_year", how="left")
cohorts.rename(columns={"scheme_name": "top_fund_preference", "amount_inr": "top_fund_invested"}, inplace=True)
cohorts

,first_transaction_year,investor_count,avg_sip_amount,total_invested,top_fund_preference,top_fund_invested
0,2024,4803,10996.885825,2258062304,Axis Small Cap Fund - Regular - Growth,65025476
1,2025,197,13505.209581,18992635,Axis Midcap Fund - Regular - Growth,1283425


## SIP Continuity Analysis

In [6]:
sip = tx[tx["is_sip"]].sort_values(["investor_id", "transaction_date"]).copy()
eligible = sip.groupby("investor_id").filter(lambda frame: len(frame) >= 6)
eligible["gap_days"] = eligible.groupby("investor_id")["transaction_date"].diff().dt.days
sip_continuity = eligible.groupby("investor_id").agg(
    sip_transactions=("transaction_date", "count"),
    avg_gap_days=("gap_days", "mean"),
    max_gap_days=("gap_days", "max"),
    total_sip_amount=("amount_inr", "sum"),
).reset_index()
sip_continuity["status"] = np.where(sip_continuity["avg_gap_days"] > 35, "at-risk", "continuous")
sip_continuity["status"].value_counts(normalize=True).rename("share")

status
at-risk       0.977974
continuous    0.022026
Name: share, dtype: float64

## Simple Fund Recommender

In [7]:
risk_map = {
    "Low": ["Low", "Low to Moderate", "Moderately Low"],
    "Moderate": ["Moderate", "Moderately High"],
    "High": ["High", "Very High"],
}

def recommend(risk_appetite):
    grades = risk_map[risk_appetite]
    table = scorecard.merge(funds[["amfi_code", "risk_category", "sub_category"]], on="amfi_code", how="left")
    return table[table["risk_category"].isin(grades)].sort_values("sharpe_ratio", ascending=False).head(3)[
        ["scheme_name", "fund_house", "sub_category", "risk_category", "sharpe_ratio", "fund_score"]
    ]

recommend("Moderate")

,scheme_name,fund_house,sub_category,risk_category,sharpe_ratio,fund_score
0,Mirae Asset Large Cap Fund - Regular - Growth,Mirae Asset MF,Large Cap,Moderate,1.448291,86.25
2,Kotak Flexicap Fund - Regular - Growth,Kotak Mahindra MF,Flexi Cap,Moderately High,1.306744,82.00
6,SBI Bluechip Fund - Regular Plan - Growth,SBI Mutual Fund,Large Cap,Moderate,1.208267,74.81


## Sector HHI Concentration

In [8]:
latest_portfolio_date = holdings["portfolio_date"].max()
hhi = (
    holdings[holdings["portfolio_date"].eq(latest_portfolio_date)]
    .assign(weight_share=lambda frame: frame["weight_pct"] / 100)
    .groupby("amfi_code", as_index=False)
    .agg(hhi=("weight_share", lambda s: float(np.square(s).sum())), holdings_count=("stock_symbol", "nunique"))
    .merge(funds[["amfi_code", "scheme_name", "category", "sub_category"]], on="amfi_code", how="left")
)
hhi = hhi[hhi["category"].str.lower().eq("equity")].sort_values("hhi", ascending=False)
hhi.head(10)

,amfi_code,hhi,holdings_count,scheme_name,category,sub_category
11,119092,0.206448,10,Axis Bluechip Fund - Regular - Growth,Equity,Large Cap
3,101207,0.200700,8,ABSL Small Cap Fund - Regular - Growth,Equity,Small Cap
18,119599,0.174751,8,SBI Small Cap Fund - Direct Plan - Growth,Equity,Small Cap
4,102885,0.174709,9,UTI Nifty 50 Index Fund - Regular - Growth,Equity,Index
7,118632,0.168298,8,Nippon India Large Cap Fund - Regular - Growth,Equity,Large Cap
29,148568,0.167930,8,Mirae Asset Emerging Bluechip Fund - Regular -...,Equity,Large & Mid Cap
21,120505,0.157570,8,ICICI Pru Midcap Fund - Regular - Growth,Equity,Mid Cap
22,120506,0.153794,9,ICICI Pru Value Discovery Fund - Regular - Growth,Equity,Value
27,125498,0.152414,8,HDFC Mid-Cap Opportunities Fund - Direct - Growth,Equity,Mid Cap
23,120841,0.149680,10,Kotak Bluechip Fund - Regular - Growth,Equity,Large Cap


## Advanced Insights

1. SBI Small Cap Fund - Direct Plan - Growth has the deepest 95% historical VaR at -2.69% daily return, making it the weakest left-tail fund in the sample.
2. ABSL Small Cap Fund - Regular - Growth has the harshest CVaR at -3.25%, so losses beyond its VaR threshold are the most severe on average.
3. The 2024 investor cohort contributes the highest total investment at INR 2,258,062,304; its preferred fund is Axis Small Cap Fund - Regular - Growth.
4. Among 1,362 investors with at least 6 SIPs, 2.2% remain continuous by the 35-day average-gap rule and 1,332 are flagged at-risk.
5. Axis Bluechip Fund - Regular - Growth is the most sector/security concentrated equity portfolio with HHI 0.206, based on the latest holdings snapshot.